## Creamos SparkContext y SarkSession

In [1]:
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pathlib import Path

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

conf = (SparkConf()
        .setMaster("local[*]")        # Corre Spark en tu máquina local, usando todos los núcleos disponibles del CPU ("local" a secas, corre Spark en modo local con un solo hilo (1 CPU))
        .setAppName("5.1 Spark y MapReduce")
        .set("spark.driver.bindAddress", "127.0.0.1")
        .set("spark.driver.host", "127.0.0.1"))
sc = SparkContext(conf = conf)



spark = SparkSession(sc)

sc, spark

(<SparkContext master=local[*] appName=5.1 Spark y MapReduce>,
 <pyspark.sql.session.SparkSession at 0x187cdf59f70>)

## CONSULTA 1
### Buscamos los 5 clientes que mas gastaron
#### Leemos el csv

In [6]:

BASE_DIR = Path.cwd()

ruta_csv = BASE_DIR.parent / "csv" / "pagos_clientes.csv"

df_payment = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

df_payment.show(5)

+-----------+------+
|customer_id|amount|
+-----------+------+
|       3330| 78.62|
|       1503| 31.08|
|        150| 28.75|
|       3207| 58.44|
|       3546| 26.35|
+-----------+------+
only showing top 5 rows


#### Aplicamos MAP SHUFFLE REDUCE

En la siguiente celda podemos apreciar el comportamiento lazy de pyspark, el computo no esta en los llamados al map, reduceByKey y sortBy. Al algoritmo le entra un rdd que es una estructura distribuidas de filas.

Sabemos que internamente pyspark no utiliza listas/arrays, pero utilizamos la notacion de lista para simplificar.

In [ ]:

rdd = df_payment.rdd

resultado = (
    rdd
    .map(lambda row: (row.customer_id, row.amount))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1] , ascending=False)
)

In [19]:
print("Tenemos un rdd, aplicamos un map para llevar a (customer_id,amount)")
print("En el reduce by key obtenemos (customer_id,[amounts]) que luego se reduce mediante la suma")
print()
print("(key, value) = (customer_id,  suma de pagos)")
print(resultado.take(5))

Tenemos un rdd, aplicamos un map para llevar a (customer_id,amount)
En el reduce by key obtenemos (customer_id,[amounts]) que luego se reduce mediante la suma

(key, value) = (customer_id,  suma de pagos)
[(3816, 854.8299999999999), (4288, 843.48), (440, 841.3199999999999), (2402, 836.78), (2109, 827.0300000000001)]


## CONSULTA 2
### Buscamos las peliculas que mas se rentan juntas (de a pares)
#### Leemos el csv

In [20]:
BASE_DIR = Path.cwd()

ruta_csv = BASE_DIR.parent / "csv" / "rental_inventory.csv"
df_ri = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

df_ri.show(5)

+---------+-------+
|rental_id|film_id|
+---------+-------+
|        1|    132|
|        1|    547|
|        1|     37|
|        1|    255|
|        1|    625|
+---------+-------+
only showing top 5 rows


#### Aplicamos MAP SHUFFLE REDUCE

In [ ]:
rdd = df_ri.rdd
print("Entra un rdd que contiene film_id y rental_id")
def combinar(iterable):
    lista =list(iterable)
    combinaciones = []
    for i in range(len(lista)-1):
        for j in range(i+1,len(lista)):
            combinaciones.append(((lista[i],lista[j]),1))
    #obtenemos [(key,value)] =[((film_id,film_id),1)]
    return combinaciones


#agrupamos las peliculas por rentas k:rental_id v:film_id
rentas = rdd.map(lambda row: (row.rental_id,row.film_id)).groupByKey()
# obtenemos k: rental_id v:film_id[]
print("Obtenemos k: renta_id v:film_id[] .")
print(rentas.take(5)) #los takes son los que ejecutan computo
print()

combinaciones = rentas.flatMap(lambda x: combinar(x[1]))
print("Obtenemos k: (film_id,film_id) v:1 .")
print(combinaciones.take(5))
print()

resultado_final = combinaciones.reduceByKey(lambda x , y: x+y).sortBy(lambda x: x[1], ascending=False)
print("Son 3 pasos: ")
print()
print("Primero se obtiene k:(film_id,film_id) v:lista de 1.")
print("Luego se obtiene k:(film_id,film_id) v:suma total.")
print("Luego se ordena por la suma total.")
print()
print("resultado_final :", resultado_final.take(10))


Obtenemos k: renta_id v:film_id[] .
[(1, <pyspark.resultiterable.ResultIterable object at 0x00000187CDED37D0>), (2, <pyspark.resultiterable.ResultIterable object at 0x00000187CFC20320>), (3, <pyspark.resultiterable.ResultIterable object at 0x00000187CFC22F60>), (4, <pyspark.resultiterable.ResultIterable object at 0x00000187CFC22750>), (5, <pyspark.resultiterable.ResultIterable object at 0x00000187CFC236E0>)]

Obtenemos k: (film_id,film_id) v:1 .
[((132, 547), 1), ((132, 37), 1), ((132, 255), 1), ((132, 625), 1), ((547, 37), 1)]

Son 3 pasos: 

Primero se obtiene k:(film_id,film_id) v:lista de 1.
Luego se obtiene k:(film_id,film_id) v:suma total.
Luego se ordena por la suma total.

resultado_final : [((277, 311), 6), ((11, 3), 5), ((615, 765), 5), ((432, 488), 5), ((226, 280), 5), ((659, 587), 5), ((208, 440), 4), ((133, 463), 4), ((723, 568), 4), ((295, 298), 4)]


## Consulta 3
### ¿Cual es el ingreso promedio por metodo de pago y cuantas transacciones tuvo cada uno?
#### Leer csv

In [23]:
BASE_DIR = Path.cwd()
ruta_csv = BASE_DIR.parent / "csv" / "pay_method_amount.csv"

df_pm_a = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

df_pm_a.show(5)

+-------------+------+
|pay_method_id|amount|
+-------------+------+
|            2| 78.62|
|            1| 31.08|
|            4| 28.75|
|            4| 58.44|
|            3| 26.35|
+-------------+------+
only showing top 5 rows


In [27]:
rdd = df_pm_a.rdd

mapeo = rdd.map(lambda row: (row.pay_method_id,(row.amount,1)))

print("Obtenemos k: pay_method_id , v: (amount,1)")
print("Mapeo: ", mapeo.take(5))
print()
pre_resultado = mapeo.reduceByKey(lambda x, y : (x[0]+y[0],x[1]+y[1]))

print("Obtenemos k: pay_method_id , v: (sum(amount),cantidad de veces que se uso)")
print("Pre resultado:", pre_resultado.take(5))
print()

resultado = pre_resultado.map(lambda tupla: (tupla[0], tupla[1][0]/tupla[1][1], tupla[1][1]))
print("Obtenemos tupla(metodo_de_pago_id,importe_promedio,cantidad de pagos realizados)")
print("resultado: ", resultado.take(5))


Obtenemos k: pay_method_id , v: (amount,1)


Mapeo:  [(2, (78.62, 1)), (1, (31.08, 1)), (4, (28.75, 1)), (4, (58.44, 1)), (3, (26.35, 1))]

Obtenemos k: pay_method_id , v: (sum(amount),cantidad de veces que se uso)
Pre resultado: [(2, (372552.43000000063, 7596)), (1, (375091.7699999995, 7455)), (4, (373113.3799999992, 7490)), (3, (368095.5399999991, 7459))]

Obtenemos tupla(metodo_de_pago_id,importe_promedio,cantidad de pagos realizados)
resultado:  [(2, 49.04587019483947, 7596), (1, 50.31412072434601, 7455), (4, 49.81487049399188, 7490), (3, 49.34918085534242, 7459)]
